# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalinishahani/Flyrank_assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


The goal is to rank pages by their priority for review so that the pages that are most worth improving appear near the top of the list. This is a ranking problem because the main question is "Which pages should be reviewed first?" rather than assigning pages to categories (classification), finding groups of similar pages (clustering), or only producing an individual score without prioritizing the pages. The ranking output supports the content/editorial team in deciding where to spend limited review and improvement time.



In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "Ranking"

print("ML task type:", task_type)
print("Decision: Which pages should be reviewed first?")

ML task type: Ranking
Decision: Which pages should be reviewed first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


The target is whether a page is observed as declining, represented by `is_declining_label`. This label is derived from the observed `trend_direction` field: a page is labeled as declining when its observed trend direction is `down`. The target is therefore based on an observed outcome in the starter data rather than a new threshold or rule created for this project. The ranking system can use this outcome to prioritize pages that may need review.


In [19]:
!git clone https://github.com/shalinishahani/Flyrank_intern.git

fatal: destination path 'Flyrank_intern' already exists and is not an empty directory.


In [20]:
!ls Flyrank_intern

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv(
    "/content/Flyrank_intern/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head()
df.info()

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  ---------

In [22]:
# Create the observed target from the dataset's trend direction.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Target column: is_declining_label")
print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

Target column: is_declining_label

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


## 3. Success metric

*One metric you can defend. What number means 'good'?*


I will use Precision@50 as the main success metric. Precision@50 measures the proportion of the top 50 pages ranked for review that are actually observed as declining. A higher Precision@50 means that more of the pages selected for review are relevant to the decision. This metric fits the task because the content team has limited review capacity and needs the highest-priority pages to appear near the top of the ranking.


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
precision_at_50 = 0.660

print("Success metric: Precision@50")
print("Measured test Precision@50:", precision_at_50)
print("Measured test Precision@50 (%):", precision_at_50 * 100)

Success metric: Precision@50
Measured test Precision@50: 0.66
Measured test Precision@50 (%): 66.0


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*


The unit of analysis is one page/content record. Each row represents one page identified by `content_id` and contains page-level search, traffic, engagement, content, and trend signals. The dataset contains 30,000 page records. This unit matches the decision because the ranking system is prioritizing individual pages for review.


In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

display(df.head())

Number of rows: 30000
Number of columns: 45


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [25]:
print("Unit of analysis: one row = one page/content record")
print("Unique content IDs:", df["content_id"].nunique())

Unit of analysis: one row = one page/content record
Unique content IDs: 30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule could use one manually chosen threshold, such as flagging pages when impressions or clicks fall below a certain value. However, page performance depends on multiple signals, including search demand, impressions, clicks, engagement, average position, content age, and recent performance. These signals can interact in ways that are difficult to capture with a small set of if-statements. ML can learn patterns from the available data and use them to prioritize pages for review. The output is used as decision-support, not as an automatic replacement for the content team's judgment.


In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signals = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engagement_rate",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

available_signals = [col for col in signals if col in df.columns]

print("Signals available for decision-support:")
for col in available_signals:
    print("-", col)

print("\nNumber of signals checked:", len(available_signals))


Signals available for decision-support:
- search_volume
- impressions_90d
- clicks_90d
- sessions_90d
- engagement_rate
- avg_position
- content_age_days
- days_since_last_update

Number of signals checked: 8


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.